In [29]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [30]:
model = pe.ConcreteModel()

### Sets

In [31]:
plane_types = ['A', 'B', 'C']
number_planes = [1, 2, 3, 4, 5]

model.plane_types = pe.Set(initialize=plane_types)
model.number_planes = pe.Set(initialize=number_planes)

### Parameters

In [32]:
cost_dict = {
    ("A", 1): 11, ("A", 2): 20, ("A", 3): 30, ("A", 4): 40, ("A", 5): 50,
    ("B", 1): 9, ("B", 2): 17, ("B", 3): 24, ("B", 4): 34, ("B", 5): 45,
    ("C", 1): 8, ("C", 2): 15, ("C", 3): 21, ("C", 4): 26, ("C", 5): 31
}
model.cost = pe.Param(model.plane_types, model.number_planes, initialize=cost_dict)
model.fixed_cost = pe.Param(initialize=6)

In [33]:
capacity_dict = {
    "A": 80, "B": 68, "C": 55
}
model.capacity = pe.Param(model.plane_types, initialize=capacity_dict)
model.number_passengers = pe.Param(initialize=372)

### Variables

In [34]:
model.x = pe.Var(model.plane_types, model.number_planes, domain=pe.Binary)

### Funcion objetivo

Minimizar el coste de los aviones para transportar a todos los pasajeros

In [35]:
def obj_rule(model):
    plane_cost = sum(
        model.cost[p, n] * model.x[p, n]
        for p in model.plane_types
        for n in model.number_planes
    )

    fixed_cost = model.fixed_cost * sum(
        model.x[p, n]
        for p in model.plane_types
        for n in model.number_planes
    )

    return plane_cost + fixed_cost

model.obj = pe.Objective(rule=obj_rule, sense=pe.minimize)

### Restricciones

In [36]:
def pasengers(model):
    return sum(model.capacity[p] * model.x[p, n] * n for p in model.plane_types for n in model.number_planes) >= model.number_passengers

model.passengers = pe.Constraint(rule=pasengers)

In [37]:
def max_type_selection(model, p):
    return sum(model.x[p, n] for n in model.number_planes) <= 1

model.max_selection = pe.Constraint(model.plane_types, rule=max_type_selection)

## Resolución con gurobi

In [38]:
solver = po.SolverFactory("gurobi_direct")

results = solver.solve(model, tee=True)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 4 rows, 15 columns and 30 nonzeros (Min)
Model fingerprint: 0x1546740f
Model has 15 linear objective coefficients
Variable types: 0 continuous, 15 integer (15 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+01, 6e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+02]

Found heuristic solution: objective 56.0000000
Presolve time: 0.00s
Presolved: 4 rows, 15 columns, 30 nonzeros
Variable types: 0 continuous, 15 integer (15 binary)

Root relaxation: objective 5.296215e+01, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work


 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0   52.96215    0    1   56.00000   52.96215  5.42%     -    0s
     0     0     cutoff    0        56.00000   56.00000  0.00%     -    0s

Cutting planes:
  Cover: 1

Explored 1 nodes (1 simplex iterations) in 0.03 seconds (0.00 work units)
Thread count was 22 (of 22 available processors)

Solution count 1: 56 

Optimal solution found (tolerance 1.00e-04)
Best objective 5.600000000000e+01, best bound 5.600000000000e+01, gap 0.0000%


In [40]:
plane_cost = sum(
    pe.value(model.cost[p, n] * model.x[p, n]) for p in model.plane_types for n in model.number_planes
)

type_cost = model.fixed_cost * sum(
    1 for i in model.plane_types for j in model.number_planes if pe.value(model.x[i, j]) == 1
)

capacity = sum(
    model.capacity[p] * pe.value(model.x[p, n]) * n for p in model.plane_types for n in model.number_planes
)

print(f"Coste de combustible: {plane_cost:.2f}k €")
print(f"Coste por cambios:   {type_cost:.2f}k €")
print(f"COSTE TOTAL MÍNIMO:  {plane_cost + type_cost:.2f}k €")
print(f"Capacidad total:     {capacity} pasajeros")

Coste de combustible: 50.00k €
Coste por cambios:   6.00k €
COSTE TOTAL MÍNIMO:  56.00k €
Capacidad total:     400.0 pasajeros
